# 07 — End-to-End Smoke Test & Submission Checklist

Run this notebook last. It verifies the final serialized model, predictions, unseen-category handling, web artifacts, and required project files.

In [1]:
from pathlib import Path
import sys, json, os

def _find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "bonbanh_usedcar_dataset.csv").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError("Project root not found. Keep notebooks inside the CarValue_AI_Complete_Project folder.")

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /mnt/data/CarValue_AI_Complete_Project


In [2]:
import json, py_compile
import numpy as np
import pandas as pd
from src.carvalue_utils import ensure_modeling_artifacts, get_train_test, load_bundle, TARGET, FEATURES

model_df, reference_year = ensure_modeling_artifacts(PROJECT_ROOT)
train, test = get_train_test(model_df)
model = load_bundle(PROJECT_ROOT / "models" / "final_model.joblib")


In [3]:
# Predict a deterministic sample from the held-out grouped test set
sample = test.sample(n=min(20, len(test)), random_state=42).copy()
pred = model.predict(sample)
assert len(pred) == len(sample)
assert np.isfinite(pred).all()
assert (pred >= 0).all()

smoke = sample[["brand","model",TARGET]].copy()
smoke["prediction_vnd"] = pred
smoke["absolute_error_vnd"] = np.abs(smoke[TARGET] - smoke["prediction_vnd"])
smoke.to_csv(PROJECT_ROOT / "artifacts" / "smoke_test_predictions.csv", index=False)
display(smoke.head(10))
print("PASS: final model returns finite non-negative prices.")


,brand,model,price_vnd,prediction_vnd,absolute_error_vnd
1427,Ford,Transit Limousine,2.300000e+08,2.925751e+08,6.257514e+07
13334,Toyota,Fortuner 2.8V 4x4 AT,8.450000e+08,8.198332e+08,2.516678e+07
2502,Kia,Carnival Premium 2.2D,9.500000e+08,9.547074e+08,4.707426e+06
6147,Honda,Civic RS 1.5 AT,6.900000e+08,7.296027e+08,3.960271e+07
19655,Mercedes-Benz,GLC 300 4Matic,1.350000e+09,1.300777e+09,4.922273e+07
1576,Honda,CRV 2.4 AT,2.900000e+08,2.676063e+08,2.239371e+07
6985,Kia,Sorento DMT,3.300000e+08,2.684745e+08,6.152554e+07
4914,Audi,Q5 2.0 AT,5.690000e+08,8.273779e+08,2.583779e+08
5513,Ford,Everest Titanium 2.0L 4x2 AT,8.050000e+08,7.660587e+08,3.894127e+07
5237,BMW,X5 xDrive40i M Sport,2.699000e+09,2.750541e+09,5.154066e+07


PASS: final model returns finite non-negative prices.


In [4]:
# Unseen category handling test
unseen = sample.iloc[[0]][FEATURES].copy()
unseen["brand"] = "__UNSEEN_BRAND_FOR_TEST__"
unseen["model"] = "__UNSEEN_MODEL_FOR_TEST__"
unseen_pred = model.predict(unseen)
assert np.isfinite(unseen_pred).all()
print("PASS: unseen category test prediction =", f"{float(unseen_pred[0]):,.0f} VND")


PASS: unseen category test prediction = 283,455,457 VND


In [5]:
# Syntax-check app without needing to launch Streamlit
py_compile.compile(str(PROJECT_ROOT / "app.py"), doraise=True)
print("PASS: app.py syntax is valid.")

required = [
    PROJECT_ROOT / "data" / "bonbanh_usedcar_dataset.csv",
    PROJECT_ROOT / "artifacts" / "CarValue_AI_model_candidate.csv",
    PROJECT_ROOT / "artifacts" / "metrics_summary.csv",
    PROJECT_ROOT / "artifacts" / "web_options.json",
    PROJECT_ROOT / "models" / "final_model.joblib",
    PROJECT_ROOT / "models" / "final_model_metadata.json",
    PROJECT_ROOT / "app.py",
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / "README.md",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Missing required deliverables: {missing}"
print("PASS: all core deliverables exist.")


PASS: app.py syntax is valid.
PASS: all core deliverables exist.


## Final checklist

- [x] EDA and data-quality audit
- [x] Repaired modeling dataset
- [x] Leakage-safe grouped split
- [x] Median + Ridge baselines
- [x] Random Forest + XGBoost + CatBoost
- [x] MAE / RMSE / R² / RMSLE
- [x] Error by price tier and major brand
- [x] Feature importance
- [x] Frozen final model
- [x] Streamlit demo app
- [x] Smoke test and serialized artifacts

**Project boundary:** the target is the online asking price, not the negotiated transaction price.